In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, sklearn, jupyterlab, ipywidgets
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas matplotlib plotly scikit-learn "jupyterlab>=3" "ipywidgets>=7.6"

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 10. EM-алгоритм та розділення суміші Гаусівських функцій

### **Це допоможе закріпити такі навички:**

- Попередньої підготовки даних до моделювання
- Роботу з бібліотеками аналізу даних

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити дані [World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness).***

Для виконання завдання необхідно виконати такі кроки:

1. **Інсталювати та імпортувати необхідні бібліотеки:** 
    - Необхідно буде інсталювати такі пакети:
	```bash
	!pip install plotly==5.20.0
	!pip install "jupyterlab>=3" "ipywidgets>=7.6"
	```

2. **Завантажити дані:**
    - З набору https://www.kaggle.com/datasets/unsdsn/world-happiness.
	```bash
	!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
	```

3. **Розпакувати дані:**
    ```bash
	!unzip WorldHappinessReport.zip
	```

4. **Прочитати дані та відобразити загальну інформацію про:**
	- Статистики
	- Типи ознак

5. **Побудувати діаграми розподілу числових ознак:**
    - Проаналізувати на відповідність чи не відповідність нормальному розподілу.

6. **Відібрати числових ознак та кореляційну матрицю:**
    - Виходячи із розуміння домену та даних відібрати певну кількість числових ознак
    - Відобразити кореляційну матрицю (*див. Тема 4. Вимірювання відстаней та подібностей в аналізі даних*)

7. **Зробити висновок про:**
    - Наявність та силу лінійного зв'язку між ознаками.

8. **Відобразити розподіл:**
    - Цільової ознаки (Happiness.Score або Happiness.Rank) за країнами.
    - Використовуючи наведений нижче код для побудови теплової мапи.
	```py
	fig = px.choropleth(data_dataframe,
						locations = "Country",
						color = "Happiness.Score",
						locationmode = "country names",
                    	)
	fig.update_layout(title = "Happiness Index 2017")
	fig.show()
	```

9. **Застосувати стандартизацію даних:**
    - Для приведення всіх значень до одного діапазону статистик.
    - Використовуючи функцію data_scale() та наступні перетворення
	```py
	def data_scale(data, scaler_type='minmax'):
	    from sklearn.preprocessing import MinMaxScaler
	    from sklearn.preprocessing import StandardScaler
	    from sklearn.preprocessing import Normalizer
	    if scaler_type == 'minmax':
	        scaler = MinMaxScaler()
	    if scaler_type == 'std':
	        scaler = StandardScaler()
	    if scaler_type == 'norm':
	        scaler = Normalizer()

	    scaler.fit(data)
	    res = scaler.transform(data)
	    return res

	data_scaled = data_scale(original_dataframe)
	df_scaled = pd.DataFrame(data_scaled, columns=[original_dataframe.columns])
	print(df_scaled.head())
	```

10. **Відобразити статистики:**
    - Отриманого стандартизованого набору даних та порівняти зі статистиками оригінального набору даних.
    - Зробити висновки.

11. **Побудувати модель кластеризації:**
	- Засобами функції `GaussianMixture()` бібліотеки `sklearn`.

12. **Побудувати теплову мапу:**
    - Для відображення розподілу країн за кластерами.

13. **Дослідити вплив:**
    - Різного набору ознак
    - Результат кластеризації

14. **Висновок:**
    - Зробити загальний висновок про відповідність результатів кластеризації оригінальному розподілу країн за ознакою.

**1. Імпорт необхідних бібліотек:**

In [2]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON (Мережа, Файлова система, Попередження)
import os
import urllib.request
import zipfile
import warnings

warnings.filterwarnings('ignore')

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import numpy as np
import pandas as pd

# 3. МАШИННЕ НАВЧАННЯ (Кластеризація та Препроцесинг)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# 4. MLOps ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

# 5. ВІЗУАЛІЗАЦІЯ ТА UI (Plotly, IPywidgets & HTML)
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("📦 Модулі архітектури імпортовано успішно!")

📦 Модулі архітектури імпортовано успішно!


**1.3. Конфігурація експерименту (Глобальні змінні):**

In [8]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА
DATASET_URL             = "https://www.kaggle.com/api/v1/datasets/download/unsdsn/world-happiness"
ZIP_PATH                = "world-happiness.zip"
TARGET_YEAR             = "2017"       # Доступні роки: "2015" | "2016" | "2017" | "2018" | "2019"
CSV_FILENAME            = f"{TARGET_YEAR}.csv"

# 2. СТРУКТУРА ДАНИХ ТА ОЗНАКИ (Вирішення проблеми Schema Drift)
SCHEMA_MAPPING = {
    "2015": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Standard Error", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2016": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Lower Confidence Interval", "Upper Confidence Interval", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2017": {
        "country": "Country",
        "target": "Happiness.Score",
        "drop": ["Happiness.Rank", "Whisker.high", "Whisker.low"],
        "features": ["Economy..GDP.per.Capita.", "Family", "Health..Life.Expectancy.", "Freedom", "Trust..Government.Corruption."]
    },
    "2018": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2019": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    }
}

CURRENT_SCHEMA          = SCHEMA_MAPPING[TARGET_YEAR]
COUNTRY_COL             = CURRENT_SCHEMA["country"]             # Динамічна колонка країни (змінювалась у 2018)
TARGET_METRIC           = CURRENT_SCHEMA["target"]              # Головна цільова метрика (Індекс щастя)
DROP_COLUMNS            = CURRENT_SCHEMA["drop"]                # Технічні колонки, що не несуть користі для кластеризації
FEATURES_FULL           = CURRENT_SCHEMA["features"]            # Повний набір соціально-економічних ознак для GMM
FEATURES_MINI           = [FEATURES_FULL[0], FEATURES_FULL[2]]  # Зменшений набір (ВВП та Здоров'я) для дослідження розмірності

# 3. МАШИННЕ НАВЧАННЯ (GMM) ТА СЕРІАЛІЗАЦІЯ
N_CLUSTERS              = 3                                     # Кількість кластерів: задає число прихованих Гаусівських розподілів (Високий, Середній, Низький рівень)
COVARIANCE_TYPE         = 'full'                                # Форма матриці коваріації (геометрія кластерів): 'full' - різні еліпси під будь-яким кутом | 'tied' - однакова форма та нахил для всіх | 'diag' - еліпси строго паралельні осям координат | 'spherical' - ідеальні круглі сфери різного радіусу
GMM_INIT_PARAMS         = 'kmeans'                              # Стратегія стартової ініціалізації (Крок 0 для EM): 'kmeans' - розумний розвідник для надійного старту | 'random' - повністю випадкові координати в просторі | 'random_from_data' - випадкові реальні точки з набору даних
SCALER_TYPE             = 'std'                                 # Алгоритм масштабування простору ознак: 'std' - центрує дисперсію навколо нуля (ідеально для GMM) | 'minmax' - жорстко стискає дані в межі від 0 до 1 | 'norm' - нормує самі вектори по їхній абсолютній довжині
N_INIT                  = 10                                    # Кількість перезапусків EM-алгоритму: захист від застрягання моделі в поганих локальних мінімумах
RANDOM_STATE            = 42                                    # Фіксація генератора псевдовипадкових чисел: гарантує 100% відтворюваність результатів експерименту

MODEL_DIR               = "GMM_Models"                          # Папка для збереження серіалізованих об'єктів
MODEL_PATH              = os.path.join(MODEL_DIR, f"gmm_{TARGET_YEAR}_{COVARIANCE_TYPE}_{GMM_INIT_PARAMS}_model.pkl") # Динамічне ім'я моделі
SCALER_PATH             = os.path.join(MODEL_DIR, f"scaler_{TARGET_YEAR}_{SCALER_TYPE}.pkl")                          # Динамічне ім'я скейлера

# 4. ВІЗУАЛІЗАЦІЯ ТА UI
PLOT_TEMPLATE           = "plotly_dark"                         # Темна тема для інтерактивних графіків Plotly
MAP_LOCATION_MODE       = "country names"                       # Режим розпізнавання країн для мап Choropleth
COLOR_SCALE_HAPPINESS   = "Viridis"                             # Безперервний градієнт для оригінального індексу щастя
COLOR_PALETTE_FULL      = px.colors.qualitative.Set1            # Контрастні дискретні кольори для 3-х кластерів (повний набір)
COLOR_PALETTE_MINI      = px.colors.qualitative.Pastel          # Пастельні кольори для експерименту зі зменшеною розмірністю

TABLE_PROPS             = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'center'}
DESCRIBE_CMAP           = 'YlGn'                                # Кольорова схема (Yellow-Green) для підсвічування описових статистик

print(f"⚙️ Глобальні константи ініціалізовано!\n   Рік: {TARGET_YEAR} | GMM({COVARIANCE_TYPE}, {GMM_INIT_PARAMS}) + {SCALER_TYPE} Scaler")

⚙️ Глобальні константи ініціалізовано!
   Рік: 2017 | GMM(full, kmeans) + std Scaler


**1.7. Приклад на HTML (Анатомія GMM):**

In [10]:
C_RAW = "#888888"                               # Базовий колір для "нерозмічених" (сирих) даних у просторі
C_C1  = COLOR_PALETTE_FULL[0]                   # Динамічний колір Кластера 1 (підтягується з глобальної палітри констант)
C_C2  = COLOR_PALETTE_FULL[1]                   # Динамічний колір Кластера 2
C_C3  = COLOR_PALETTE_FULL[2]                   # Динамічний колір Кластера 3

html_em_pipeline = f"""
<div style="font-family: sans-serif; max-width: 900px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333; margin: auto;">
    <h2 style="color: #00c3ff; text-align: center; margin-top: 0;">🧠 Анатомія GMM: Що робить EM-алгоритм з країнами?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {C_RAW}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий простір (Дані після {SCALER_TYPE} Scaler)</div>
        <div style="color: {C_RAW}; font-size: 15px; margin-top: 5px; font-style: italic;">
            Маємо N країн у багатовимірному просторі ознак (ВВП, Здоров'я, Свобода...).<br>
            Усі точки "сірі", алгоритм ще нічого не знає про кластери.
        </div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Ініціалізація (Метод '{GMM_INIT_PARAMS}')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px;">
            ШІ генерує {N_CLUSTERS} випадкові багатовимірні "дзвони" (Гаусівські розподіли).<br>
            Кожен має свій центр <b>(μ)</b> та матрицю коваріації <b>(Σ)</b>.
        </div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇ ♻️ Цикл EM-алгоритму ♻️ ⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: E-крок (Expectation / Очікування)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px;">
            Обчислення м'якої ймовірності (Soft Clustering) за формулою Баєса:<br>
            <i>"Країна Х належить до Кластера-1 на 10%, Кластера-2 на 85%, Кластера-3 на 5%".</i>
        </div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: M-крок (Maximization / Максимізація)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px;">
            Оновлення параметрів дзвонів: <b>Нові μ</b> тягнуться до скупчень точок, <b>Нові Σ</b> змінюють форму еліпсів.
        </div>
    </div>

    <div style="text-align: center; color: #00ffcc; font-size: 20px; margin-top: 10px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed #00ffcc; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Збіжність (Convergence)</div>
        <div style="color: #00ffcc; font-size: 18px; margin-top: 10px; font-family: monospace;">[ <span style="color:{C_C1}">Кластер 1</span> | <span style="color:{C_C2}">Кластер 2</span> | <span style="color:{C_C3}">Кластер 3</span> ]</div>
    </div>
</div>
"""

print("Красивий Вивід (Інтерактивна схема логіки алгоритму):")
display(HTML(html_em_pipeline))

Красивий Вивід (Інтерактивна схема логіки алгоритму):


**2. Завантажити дані:**

**3. Розпакувати дані:**

**4. Прочитати дані та відобразити загальну інформацію:**

**5. Побудувати діаграми розподілу числових ознак:**

**6. Відібрати числових ознак та кореляційну матрицю:**

**7. Зробити висновок:**

**8. Відобразити розподіл:**

**9. Застосувати стандартизацію даних:**

**10. Відобразити статистики:**

**11. Побудувати модель кластеризації:**

**12. Побудувати теплову мапу:**

**13. Дослідити вплив:**

**13.5.\*\* Експорт навченої моделі ШІ:**

**13.9.\*\* Класифікація щастя:**

**14. Висновок:**